In [0]:
silver_configs = spark.table("dev.demo.silver_config").collect()

In [0]:
datasets = {}

for config in silver_configs:
    table_name = config["table_name"]

    datasets[table_name] = spark.read.format("delta").load(
        f"/Volumes/dev/demo/datasets/bronze/{table_name}"
    )

In [0]:
for config in silver_configs:

    table_name = config["table_name"]
    primary_key = config["primary_key"]

    datasets[table_name] = (
        datasets[table_name]
        .dropDuplicates([primary_key])
    )

In [0]:
orders_df = datasets["orders"]
customers_df = datasets["customers"]
products_df = datasets["products"]
product_category_df = datasets["product_category"]

In [0]:
# join all datasets
# First flatten the nested struct columns from orders_df
from pyspark.sql.functions import col

orders_flat = orders_df.select(
    col("customer.customer_id").alias("customer_id"),
    col("customer.customer_name").alias("customer_name"),
    col("product.product_id").alias("product_id"),
    col("product.product_name").alias("product_name"),
    "order_id", "order_date", "quantity", "ingestion_ts", "source_file"
)

# Drop duplicate columns from right-side DataFrames to avoid AMBIGUOUS_REFERENCE
silver_df = (
    orders_flat
    .join(customers_df.drop("customer_name", "ingestion_ts", "source_file"), on="customer_id", how="left")
    .join(products_df.drop("product_name", "ingestion_ts", "source_file"), on="product_id", how="left")
    .join(product_category_df.drop("ingestion_ts", "source_file"), on="category_id", how="left")
)

In [0]:
# display(silver_df)

In [0]:
## Business Transformations
from pyspark.sql.functions import *

silver_df = (silver_df.withColumn("sales_amount", col("quantity") * col("price"))
    .withColumn("sales_month", date_format("order_date", "yyyy-MM"))
)

In [0]:
from pyspark.sql.window import Window

##Window Functions
customer_window = Window.partitionBy("customer_id").orderBy("order_date")

## Customer order sequence:
silver_df = silver_df.withColumn("customer_order_seq", row_number().over(customer_window))

## Previous purchase amount:
silver_df = silver_df.withColumn("previous_sales", lag("sales_amount").over(customer_window))

##Running total:
silver_df = silver_df.withColumn("running_total_sales", sum("sales_amount").over(customer_window))

In [0]:
## Select Business Columns

silver_final_df = silver_df.select(
    "order_id",
    "order_date",
    "sales_month",
    "customer_id",
    "customer_name",
    "product_id",
    "product_name",
    "category_name",
    "department",
    "quantity",
    "sales_amount",
    "customer_order_seq",
    "previous_sales",
    "running_total_sales"
)



In [0]:
# display(silver_final_df)

In [0]:
## write into delta

silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/dev/demo/datasets/silver/orders_sales")

In [0]:
## writing into table
silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev.demo.fact_sales")

In [0]:
%sql
select * from dev.demo.fact_sales

order_id,order_date,sales_month,customer_id,customer_name,product_id,product_name,category_name,department,quantity,sales_amount,customer_order_seq,previous_sales,running_total_sales
1,2025-01-10,2025-01,101,Anil,1001,Laptop,Electronics,Technology,1,50000,1,null,50000
4,2025-04-13,2025-04,101,Anil,1004,Monitor,Electronics,Technology,1,12000,2,50000,62000
5,2025-05-14,2025-05,101,Anil,1002,Mobile,Electronics,Technology,2,50000,3,12000,112000
8,2025-08-17,2025-08,101,Anil,1003,Headphones,Accessories,Technology,4,8000,4,50000,120000
9,2025-09-10,2025-09,101,Anil,1001,Laptop,Electronics,Technology,1,50000,5,8000,170000
2,2025-02-11,2025-02,102,Rahul,1002,Mobile,Electronics,Technology,2,50000,1,null,50000
6,2025-06-15,2025-06,102,Rahul,1001,Laptop,Electronics,Technology,1,50000,2,50000,100000
10,2025-09-15,2025-09,102,Rahul,1004,Monitor,Electronics,Technology,2,24000,3,50000,124000
3,2025-03-12,2025-03,103,Priya,1003,Headphones,Accessories,Technology,3,6000,1,null,6000
7,2025-07-16,2025-07,103,Priya,1004,Monitor,Electronics,Technology,2,24000,2,6000,30000


In [0]:
spark.read.format("delta").load(
    "/Volumes/dev/demo/datasets/bronze/orders"
).select("order_id").distinct().count()

13